# SC-6-Errors-Events - Erreurs et Evenements

[<< Inheritance](SC-5-Inheritance.ipynb) | [Token Standards >>](../02-Solidity-Advanced/SC-7-Token-Standards.ipynb)

---

## Objectifs d'apprentissage

1. Gerer les erreurs avec `require`, `revert`, `assert`
2. Creer des **custom errors** (Solidity 0.8.4+)
3. Emettre des **events** pour le logging

### Prerequis

- [SC-1](SC-3-Solidity-Basics.ipynb) a [SC-3](SC-5-Inheritance.ipynb) completes
- Comprendre les transactions Ethereum (gas, reverts)

### Duree estimee : 30 minutes

---

## 0. Connexion a la blockchain locale

Tous les contrats de ce notebook sont **compiles et deployes reellement** sur anvil.
Lancez `anvil` dans un terminal avant d'executer les cellules.


In [1]:
# Connection a anvil (blockchain locale Foundry)
# Prerequis: anvil en cours d'execution dans un terminal
from web3 import Web3
import solcx

SOLC_VERSION = "0.8.28"
ANVIL_URL = "http://127.0.0.1:8545"

# Connexion
w3 = Web3(Web3.HTTPProvider(ANVIL_URL))
assert w3.is_connected(), f"Impossible de se connecter a {ANVIL_URL}. Lancez 'anvil' dans un terminal."

# Installer solc si necessaire
installed = [str(v) for v in solcx.get_installed_solc_versions()]
if SOLC_VERSION not in installed:
    solcx.install_solc(SOLC_VERSION)
solcx.set_solc_version(SOLC_VERSION)

deployer = w3.eth.accounts[0]
print(f"Connecte a anvil (chain {w3.eth.chain_id}), deployer: {deployer[:10]}...")


def compile_and_deploy(w3, source_code, deployer, *constructor_args):
    """Compiler et deployer un contrat Solidity (source a un seul contrat)."""
    compiled = solcx.compile_source(
        source_code, output_values=["abi", "bin"], solc_version=SOLC_VERSION
    )
    contract_id, contract_interface = compiled.popitem()
    Contract = w3.eth.contract(
        abi=contract_interface["abi"], bytecode=contract_interface["bin"]
    )
    tx_hash = Contract.constructor(*constructor_args).transact({"from": deployer})
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
    instance = w3.eth.contract(
        address=receipt.contractAddress, abi=contract_interface["abi"]
    )
    print(f"Deploye: {contract_id.split(':')[-1]} a {receipt.contractAddress}")
    return instance, receipt


def deploy_named(w3, source_code, contract_name, deployer, *constructor_args):
    """Compiler une source multi-contrats et deployer un contrat specifique par nom."""
    compiled = solcx.compile_source(
        source_code, output_values=["abi", "bin"], solc_version=SOLC_VERSION
    )
    for contract_id, contract_interface in compiled.items():
        if contract_id.split(':')[-1] == contract_name:
            Contract = w3.eth.contract(
                abi=contract_interface["abi"], bytecode=contract_interface["bin"]
            )
            tx_hash = Contract.constructor(*constructor_args).transact({"from": deployer})
            receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
            instance = w3.eth.contract(
                address=receipt.contractAddress, abi=contract_interface["abi"]
            )
            print(f"Deploye: {contract_name} a {receipt.contractAddress}")
            return instance, receipt
    raise ValueError(f"Contrat '{contract_name}' introuvable dans la source compilee")

Connecte a anvil (chain 31337), deployer: 0xf39Fd6e5...


---

## 1. Gestion des erreurs

| Fonction | Usage | Gas remboursé |
|----------|-------|---------------|
| `require` | Conditions d'entree | Oui |
| `revert` | Erreurs complexes | Oui |
| `assert` | Invariants internes | Non |

In [2]:
# Gestion des erreurs
ERROR_HANDLING = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract ErrorHandling {
    address public owner;
    uint256 public value;

    constructor() {
        owner = msg.sender;
    }

    function setValue(uint256 _value) public {
        require(msg.sender == owner, "Not owner");
        require(_value > 0, "Value must be positive");
        value = _value;
    }

    function complexCheck(uint256 a, uint256 b) public pure {
        if (a == 0) { revert("A cannot be zero"); }
        if (b == 0) { revert("B cannot be zero"); }
        if (a + b > 100) { revert("Sum too large"); }
    }

    function invariant() public view {
        assert(owner != address(0));
    }
}
'''

print("--- require / revert / assert ---")
eh, _ = compile_and_deploy(w3, ERROR_HANDLING, deployer)
print(f"  Deploye : {eh.address}")

# require : not owner
other = w3.eth.accounts[1]
try:
    eh.functions.setValue(42).transact({'from': other})
except Exception:
    print("  setValue(42) par other → revert 'Not owner' (attendu)")

# require : value must be positive
try:
    eh.functions.setValue(0).transact({'from': deployer})
except Exception:
    print("  setValue(0) → revert 'Value must be positive' (attendu)")

# require : success
eh.functions.setValue(42).transact({'from': deployer})
print(f"  setValue(42) par owner → value = {eh.functions.value().call()}")

# revert : sum too large
try:
    eh.functions.complexCheck(60, 50).transact({'from': deployer})
except Exception:
    print("  complexCheck(60, 50) → revert 'Sum too large' (attendu)")

# assert : invariant holds
eh.functions.invariant().call()
print("  invariant() → passe (owner != address(0))")

--- require / revert / assert ---
Deploye: ErrorHandling a 0x0355B7B8cb128fA5692729Ab3AAa199C1753f726
  Deploye : 0x0355B7B8cb128fA5692729Ab3AAa199C1753f726
  setValue(42) par other → revert 'Not owner' (attendu)
  setValue(0) → revert 'Value must be positive' (attendu)
  setValue(42) par owner → value = 42
  complexCheck(60, 50) → revert 'Sum too large' (attendu)
  invariant() → passe (owner != address(0))


---

## 2. Custom Errors (Solidity 0.8.4+)

Les custom errors sont plus economiques en gas que les strings.

In [3]:
# Custom errors
CUSTOM_ERRORS = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

error InsufficientBalance(uint256 available, uint256 required);
error Unauthorized(address caller);
error InvalidAmount();

contract CustomErrors {
    mapping(address => uint256) public balances;

    function deposit() public payable {
        balances[msg.sender] += msg.value;
    }

    function withdraw(uint256 amount) public {
        if (balances[msg.sender] < amount) {
            revert InsufficientBalance(balances[msg.sender], amount);
        }
        balances[msg.sender] -= amount;
        payable(msg.sender).transfer(amount);
    }

    function transfer(address to, uint256 amount) public {
        if (amount == 0) {
            revert InvalidAmount();
        }
        if (balances[msg.sender] < amount) {
            revert InsufficientBalance(balances[msg.sender], amount);
        }
        balances[msg.sender] -= amount;
        balances[to] += amount;
    }
}
'''

print("--- Custom errors avec parametres ---")
ce, _ = compile_and_deploy(w3, CUSTOM_ERRORS, deployer)
receiver = w3.eth.accounts[1]

# Deposit 0.5 ETH
ce.functions.deposit().transact({'from': deployer, 'value': w3.to_wei(0.5, 'ether')})
print(f"  Apres deposit(0.5 ETH) : balance = {w3.from_wei(ce.functions.balances(deployer).call(), 'ether')} ETH")

# withdraw trop → InsufficientBalance
try:
    ce.functions.withdraw(w3.to_wei(1, 'ether')).transact({'from': deployer})
except Exception as e:
    print(f"  withdraw(1 ETH) → revert InsufficientBalance (attendu)")

# transfer(0) → InvalidAmount
try:
    ce.functions.transfer(receiver, 0).transact({'from': deployer})
except Exception:
    print("  transfer(0) → revert InvalidAmount (attendu)")

# transfer valide
ce.functions.transfer(receiver, w3.to_wei(0.1, 'ether')).transact({'from': deployer})
print(f"  Apres transfer(0.1 ETH) : receiver balance = {w3.from_wei(ce.functions.balances(receiver).call(), 'ether')} ETH")

--- Custom errors avec parametres ---
Deploye: CustomErrors a 0xf4B146FbA71F41E0592668ffbF264F1D186b2Ca8
  Apres deposit(0.5 ETH) : balance = 0.5 ETH
  withdraw(1 ETH) → revert InsufficientBalance (attendu)


  transfer(0) → revert InvalidAmount (attendu)


  Apres transfer(0.1 ETH) : receiver balance = 0.1 ETH


---

## 3. Events

Les events permettent de logger des donnees sur la blockchain.

In [4]:
# Events
EVENTS_EXAMPLE = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract EventsExample {
    event Deposit(address indexed from, uint256 amount, uint256 timestamp);
    event Withdrawal(address indexed to, uint256 amount);
    event Transfer(address indexed from, address indexed to, uint256 amount);
    event OwnershipTransferred(address indexed previousOwner, address indexed newOwner);

    address public owner;
    mapping(address => uint256) public balances;

    constructor() {
        owner = msg.sender;
    }

    function deposit() public payable {
        balances[msg.sender] += msg.value;
        emit Deposit(msg.sender, msg.value, block.timestamp);
    }

    function withdraw(uint256 amount) public {
        require(balances[msg.sender] >= amount, "Insufficient balance");
        balances[msg.sender] -= amount;
        payable(msg.sender).transfer(amount);
        emit Withdrawal(msg.sender, amount);
    }

    function transferFunds(address to, uint256 amount) public {
        require(balances[msg.sender] >= amount, "Insufficient balance");
        balances[msg.sender] -= amount;
        balances[to] += amount;
        emit Transfer(msg.sender, to, amount);
    }

    function transferOwnership(address newOwner) public {
        require(msg.sender == owner, "Not owner");
        require(newOwner != address(0), "Invalid address");
        address previousOwner = owner;
        owner = newOwner;
        emit OwnershipTransferred(previousOwner, newOwner);
    }
}
'''

print("--- Events : Deposit, Transfer, OwnershipTransferred ---")
ev, _ = compile_and_deploy(w3, EVENTS_EXAMPLE, deployer)
receiver = w3.eth.accounts[1]

# Deposit → event Deposit
tx = ev.functions.deposit().transact({'from': deployer, 'value': w3.to_wei(1, 'ether')})
receipt = w3.eth.get_transaction_receipt(tx)
logs = ev.events.Deposit().process_receipt(receipt)
print(f"  Deposit event : from={logs[0]['args']['from'][:10]}..., amount={w3.from_wei(logs[0]['args']['amount'], 'ether')} ETH")

# transferFunds → event Transfer
tx = ev.functions.transferFunds(receiver, w3.to_wei(0.3, 'ether')).transact({'from': deployer})
receipt = w3.eth.get_transaction_receipt(tx)
logs = ev.events.Transfer().process_receipt(receipt)
print(f"  Transfer event : from={logs[0]['args']['from'][:10]}..., to={logs[0]['args']['to'][:10]}..., amount={w3.from_wei(logs[0]['args']['amount'], 'ether')} ETH")

# transferOwnership → event OwnershipTransferred
new_owner = w3.eth.accounts[2]
tx = ev.functions.transferOwnership(new_owner).transact({'from': deployer})
receipt = w3.eth.get_transaction_receipt(tx)
logs = ev.events.OwnershipTransferred().process_receipt(receipt)
print(f"  OwnershipTransferred : previous={logs[0]['args']['previousOwner'][:10]}..., new={logs[0]['args']['newOwner'][:10]}...")

--- Events : Deposit, Transfer, OwnershipTransferred ---
Deploye: EventsExample a 0xBEc49fA140aCaA83533fB00A2BB19bDdd0290f25
  Deposit event : from=0xf39Fd6e5..., amount=1 ETH
  Transfer event : from=0xf39Fd6e5..., to=0x70997970..., amount=0.3 ETH


  OwnershipTransferred : previous=0xf39Fd6e5..., new=0x3C44CdDd...


### 3.1 Indexed parameters

- Maximum **3 parametres indexes** par event
- Les parametres indexes sont recherchables dans les logs
- Les parametres non indexes sont stockes dans `data`

In [5]:
# Indexed parameters
INDEXED_EXAMPLE = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract IndexedExample {
    event OrderCreated(
        uint256 indexed orderId,
        address indexed buyer,
        address indexed seller,
        uint256 amount,
        string productId
    );

    function createOrder(
        address seller,
        uint256 amount,
        string memory productId
    ) public returns (uint256) {
        uint256 orderId = uint256(keccak256(abi.encodePacked(
            msg.sender, seller, amount, block.timestamp
        )));
        emit OrderCreated(orderId, msg.sender, seller, amount, productId);
        return orderId;
    }
}
'''

print("--- Indexed parameters : recherchables dans les logs ---")
idx, _ = compile_and_deploy(w3, INDEXED_EXAMPLE, deployer)
seller = w3.eth.accounts[1]

tx = idx.functions.createOrder(seller, 500, "PROD-001").transact({'from': deployer})
receipt = w3.eth.get_transaction_receipt(tx)
logs = idx.events.OrderCreated().process_receipt(receipt)
args = logs[0]['args']
print(f"  orderId (indexed) = {hex(args['orderId'])[:18]}...")
print(f"  buyer   (indexed) = {args['buyer'][:10]}...")
print(f"  seller  (indexed) = {args['seller'][:10]}...")
print(f"  amount  (data)    = {args['amount']}")
print(f"  productId (data)  = '{args['productId']}'")
print("  Les 3 premiers champs sont indexables par filtrage, les 2 derniers sont dans data")

--- Indexed parameters : recherchables dans les logs ---
Deploye: IndexedExample a 0x46b142DD1E924FAb83eCc3c08e4D46E82f005e0E


  orderId (indexed) = 0x3876973c5e96c0fe...
  buyer   (indexed) = 0xf39Fd6e5...
  seller  (indexed) = 0x70997970...
  amount  (data)    = 500
  productId (data)  = 'PROD-001'
  Les 3 premiers champs sont indexables par filtrage, les 2 derniers sont dans data


---

## 4. Try/Catch pour appels externes

In [6]:
# Try/Catch
TRY_CATCH_SOURCE = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

// Contrat externe controllable pour la demo
contract MockExternal {
    bool public shouldFail = true;

    function riskyOperation() external returns (bool) {
        require(!shouldFail, "Operation failed by design");
        return true;
    }

    function setFail(bool _fail) external {
        shouldFail = _fail;
    }
}

interface IExternalContract {
    function riskyOperation() external returns (bool);
}

contract TryCatchExample {
    event OperationSuccess(bool result);
    event OperationFailed(string reason);

    IExternalContract public externalContract;

    constructor(address _contract) {
        externalContract = IExternalContract(_contract);
    }

    function safeOperation() public returns (bool) {
        try externalContract.riskyOperation() returns (bool success) {
            emit OperationSuccess(success);
            return success;
        } catch Error(string memory reason) {
            emit OperationFailed(reason);
            return false;
        } catch {
            emit OperationFailed("Unknown error");
            return false;
        }
    }
}
'''

print("--- Try/Catch pour appels externes ---")

# Deployer MockExternal en premier
mock, _ = deploy_named(w3, TRY_CATCH_SOURCE, "MockExternal", deployer)

# Deployer TryCatchExample en pointant vers MockExternal
tce, _ = deploy_named(w3, TRY_CATCH_SOURCE, "TryCatchExample", deployer, mock.address)
print(f"  MockExternal : shouldFail = {mock.functions.shouldFail().call()}")

# safeOperation() quand shouldFail=True → catch Error
tx = tce.functions.safeOperation().transact({'from': deployer})
receipt = w3.eth.get_transaction_receipt(tx)
failed_logs = tce.events.OperationFailed().process_receipt(receipt)
print(f"  safeOperation() [shouldFail=True] → OperationFailed: '{failed_logs[0]['args']['reason']}'")

# Changer shouldFail a False
mock.functions.setFail(False).transact({'from': deployer})

# safeOperation() quand shouldFail=False → success
tx = tce.functions.safeOperation().transact({'from': deployer})
receipt = w3.eth.get_transaction_receipt(tx)
success_logs = tce.events.OperationSuccess().process_receipt(receipt)
print(f"  safeOperation() [shouldFail=False] → OperationSuccess: {success_logs[0]['args']['result']}")

--- Try/Catch pour appels externes ---


Deploye: MockExternal a 0x1c85638e118b37167e9298c2268758e058DdfDA0
Deploye: TryCatchExample a 0x367761085BF3C12e5DA2Df99AC6E1a824612b8fb
  MockExternal : shouldFail = True


  safeOperation() [shouldFail=True] → OperationFailed: 'Operation failed by design'
  safeOperation() [shouldFail=False] → OperationSuccess: True


---

## 5. Exercices

### Exercice 1 : Contrat avec custom errors

Creez un contrat de vote avec custom errors pour les cas invalides.

In [7]:
# Exercice 1 : Contrat de vote avec custom errors
EXERCICE_VOTING = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

// Custom errors
error AlreadyVoted(address voter);
error VotingClosed();
error InvalidProposal(uint256 proposalId);

contract Voting {
    struct Proposal {
        string name;
        uint256 voteCount;
    }

    Proposal[] public proposals;
    mapping(address => bool) public hasVoted;
    bool public votingOpen = true;

    event Voted(address indexed voter, uint256 indexed proposalId);
    event VotingClosed(uint256 timestamp);

    constructor(string[] memory _proposalNames) {
        for (uint i = 0; i < _proposalNames.length; i++) {
            proposals.push(Proposal(_proposalNames[i], 0));
        }
    }

    function vote(uint256 proposalId) public {
        // TODO: Verifier que le vote est ouvert (sinon revert VotingClosed)
        // TODO: Verifier que l utilisateur n a pas deja vote (sinon revert AlreadyVoted)
        // TODO: Verifier que le proposalId est valide (sinon revert InvalidProposal)
        // TODO: Marquer l utilisateur comme ayant vote
        // TODO: Incrementer le compteur de votes de la proposition
        // TODO: Emettre l evenement Voted
    }

    function closeVoting() public {
        // TODO: Fermer le vote
        // TODO: Emettre l evenement VotingClosed avec le timestamp
    }
}
'''

# Deployer (ajuster les arguments du constructeur si necessaire)
# voting, receipt = compile_and_deploy(w3, EXERCICE_VOTING, deployer)
# print(f"Deploye a: {voting.address}")

### Exercice 2 : Contrat Auction avec events

Creez un contrat d'enchere avec events pour chaque action.

In [8]:
# Exercice 2 : Contrat Auction avec events
EXERCICE_AUCTION = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

error AuctionEnded();
error BidTooLow(uint256 currentBid, uint256 minBid);

contract Auction {
    address public beneficiary;
    uint256 public auctionEndTime;
    address public highestBidder;
    uint256 public highestBid;
    bool public ended;

    mapping(address => uint256) public pendingReturns;

    event HighestBidIncreased(address indexed bidder, uint256 amount);
    event AuctionFinalized(address winner, uint256 amount);
    event BidRefunded(address indexed bidder, uint256 amount);

    constructor(uint256 biddingTime, address _beneficiary) {
        beneficiary = _beneficiary;
        auctionEndTime = block.timestamp + biddingTime;
    }

    function bid() public payable {
        // TODO: Verifier que l enchere n est pas terminee (sinon revert AuctionEnded)
        // TODO: Verifier que l offre est superieure a l offre actuelle (sinon revert BidTooLow)
        // TODO: Rembourser l ancien meilleur encherisseur via pendingReturns
        // TODO: Mettre a jour highestBidder et highestBid
        // TODO: Emettre HighestBidIncreased
    }

    function withdraw() public returns (bool) {
        // TODO: Recuperer le montant en attente pour msg.sender
        // TODO: Si montant > 0, remettre a zero et transferer
        // TODO: Emettre BidRefunded
    }

    function auctionEnd() public {
        // TODO: Verifier que le temps est ecoule
        // TODO: Verifier que l enchere n est pas deja finalisee
        // TODO: Marquer comme finalisee
        // TODO: Emettre AuctionFinalized
        // TODO: Transferer les fonds au beneficiaire
    }
}
'''

# Deployer (ajuster les arguments du constructeur si necessaire)
# auction, receipt = compile_and_deploy(w3, EXERCICE_AUCTION, deployer)
# print(f"Deploye a: {auction.address}")

---

## 6. Resume

| Concept | Description | Gas |
|---------|-------------|-----|
| `require` | Conditions d'entree | Rembourse |
| `revert` | Erreurs complexes | Rembourse |
| `assert` | Invariants internes | Non rembourse |
| `error` | Custom errors (0.8.4+) | Economique |
| `event` | Logging blockchain | - |
| `indexed` | Parametres recherchables | Max 3 |

---

**Notebook suivant** : [SC-7-Token-Standards](../02-Solidity-Advanced/SC-7-Token-Standards.ipynb)

---

[<< Inheritance](SC-5-Inheritance.ipynb) | [Token Standards >>](../02-Solidity-Advanced/SC-7-Token-Standards.ipynb)